# Problem Set 3 — Watching a disk evolve
*ASTR 5820*

This notebook turns the viscous evolver you wrote in `step()` into figures and animations of two things: where a disk's mass goes, and where its angular momentum goes. The plotting and animation code is written for you. You fill in the cells marked **TODO**, eight short pieces, most of them a few lines, and answer three questions in one or two sentences each.

Before you start, `scale_height`, `sed` and `step` must be working; the notebook calls your `step()` throughout.

$$
\frac{\partial\Sigma}{\partial t} = \frac{3}{r}\frac{\partial}{\partial r}\left[r^{1/2}\frac{\partial}{\partial r}\left(\nu\Sigma r^{1/2}\right)\right],
\qquad
\dot M(r) = 6\pi r^{1/2}\frac{\partial}{\partial r}\left(\nu\Sigma r^{1/2}\right)\ \ (\text{positive inward}).
$$

The complete notebook runs in about two minutes. Each animation is saved to `figures/` as an MP4 and embedded below; without `ffmpeg` it is embedded frame by frame instead. Submit the notebook with its outputs.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Video

from astrotools import constants as c
from astrotools.disk import structure as s
from astrotools.disk import viscous as v

step = v.step

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e2dc"
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 11, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "animation.embed_limit": 60,
})
os.makedirs("figures", exist_ok=True)

def show(anim, name, fps):
    # Save figures/<name>.mp4 and embed it; without ffmpeg, embed the frames instead.
    if animation.writers.is_available("ffmpeg"):
        path = f"figures/{name}.mp4"
        anim.save(path, writer=animation.FFMpegWriter(fps=fps, bitrate=1800), dpi=100)
        return Video(path, embed=True, html_attributes="controls loop")
    return HTML(anim.to_jshtml(fps=fps, default_mode="once"))

**TODO 1.** The animations show the mass flow through the disk, so you need $\dot M$ at every cell face. It is the flux your `step()` already computes.

In [ ]:
def mdot_faces(sigma, grid, nu):
    # Mass flow through each of the n + 1 cell faces [kg/s], positive inward.
    # TODO: return the mass flow through every cell face, positive inward,
    #       Mdot = 6 pi r_face^(1/2) d(nu Sigma r^(1/2))/dr  -- the same flux your step() uses.
    raise NotImplementedError("TODO: mdot_faces")

## 1. Why mass must move inward

Two rings of mass $m_1$ and $m_2$ orbit at $r_1 < r_2$ and trade angular momentum only with each other. With $s_i \equiv r_i^{1/2}$,

$$
J = \sqrt{GM_\star}\,(m_1 s_1 + m_2 s_2), \qquad E = -\frac{GM_\star}{2}\left(\frac{m_1}{s_1^2} + \frac{m_2}{s_2^2}\right).
$$

Move the inner ring by $\delta r_1$, let the outer ring move however it must to keep $J$ fixed, and see what happens to $E$.

**TODO 2.**

In [ ]:
GM = c.GM_SUN
m1 = m2 = 1.0
r1, r2 = 5 * c.AU, 10 * c.AU
J0 = np.sqrt(GM) * (m1 * np.sqrt(r1) + m2 * np.sqrt(r2))
E0 = -GM / 2 * (m1 / r1 + m2 / r2)
dr1 = np.linspace(-1.5, 1.5, 301) * c.AU
r1n = r1 + dr1

# TODO: for each displacement of the inner ring, find the outer ring's radius r2n that keeps
#       J fixed, and the total energy E of the pair. Use the expressions above (s = r^(1/2)).
r2n = ...
E = ...
raise NotImplementedError("TODO: two rings")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(dr1 / c.AU, (E - E0) / abs(E0), color=BLUE)
ax[0].axhline(0, color=INK2, lw=0.8); ax[0].axvline(0, color=INK2, lw=0.8)
ax[0].set_xlabel(r"$\delta r_1$ [AU]  (inner ring)"); ax[0].set_ylabel(r"$\Delta E / |E_0|$ at fixed $J$")
ax[1].plot(dr1 / c.AU, (r2n - r2) / c.AU, color=ORANGE)
ax[1].axhline(0, color=INK2, lw=0.8); ax[1].axvline(0, color=INK2, lw=0.8)
ax[1].set_xlabel(r"$\delta r_1$ [AU]"); ax[1].set_ylabel(r"$\delta r_2$ [AU]  (outer ring)")
fig.suptitle("Two rings at 5 and 10 AU exchanging angular momentum", color=INK, fontsize=11)
fig.tight_layout()

## 2. A spreading ring

A ring at $r_0 = 10$ AU spreads with constant $\nu$, from $\tau = 0.004$ to 10, where $\tau = 12\nu t/r_0^2$. The animation shows mass and angular momentum per unit $\ln r$ (top) and the mass flow normalized to its largest value at each moment (bottom), with the stagnation radius, where the flow reverses, marked.

**TODO 3.** Fill in the time stepping.

In [ ]:
r0, m_ring, nu0 = 10 * c.AU, 0.01 * c.M_SUN, 1.0e11
t_nu = r0 ** 2 / (12 * nu0)
g = v.make_grid(0.5 * c.AU, 3000 * c.AU, 113)                 # 30 cells per decade
nu_ring = np.full(g.n, nu0)
tau0, tau_end, n_frames = 0.004, 10.0, 100
taus = np.geomspace(tau0, tau_end, n_frames)
dt = v.cfl_timestep(g, nu_ring)

sig = v.ring_solution(g.r, tau0 * t_nu, m_ring, r0, nu0)
frames = [sig.copy()]
t_now = tau0 * t_nu
for tk in taus[1:]:
    # TODO: advance sig with your step() until t_now reaches tk * t_nu, using steps of at most dt
    #       and shortening the last one so the snapshot lands exactly on tk * t_nu.
    raise NotImplementedError("TODO: ring evolution")
    frames.append(sig.copy())
frames = np.array(frames)
j = np.sqrt(c.G * c.M_SUN * g.r)
M0, J0 = (frames[0] * g.area).sum(), (frames[0] * g.area * j).sum()

**TODO 4** (the profiles) and **TODO 5** (the stagnation radius, inside `draw`).

In [ ]:
x = g.r / c.AU
# TODO: mass and angular momentum per unit ln r for a surface-density profile S, each divided by
#       its initial total (M0, J0). Mass per ln r is 2 pi r^2 Sigma; multiply by j for angular momentum.
dMdlnr = lambda S: ...
dJdlnr = lambda S: ...
raise NotImplementedError("TODO: per-ln r profiles")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8.0, 6.4), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
lM, = a1.semilogx(x, dMdlnr(frames[0]), color=BLUE, label=r"mass, $dM/d\ln r$")
lJ, = a1.semilogx(x, dJdlnr(frames[0]), color=ORANGE, label=r"angular momentum, $dJ/d\ln r$")
lA, = a1.semilogx(x, dMdlnr(v.ring_solution(g.r, tau0 * t_nu, m_ring, r0, nu0)), ":", color=INK, lw=1.2,
                  label="analytic (mass)")
a1.set_yscale("log"); a1.set_ylim(1e-4, 50); a1.set_xlim(1, 1000)
a1.set_ylabel("per unit ln r\n(normalized to initial total)"); a1.legend(loc="upper right", fontsize=9)
txt = a1.text(0.02, 0.92, "", transform=a1.transAxes, color=INK)
fr = a1.text(0.02, 0.80, "", transform=a1.transAxes, color=INK2, fontsize=10)
F0 = mdot_faces(frames[0], g, nu_ring)
lF, = a2.semilogx(g.r_face / c.AU, F0 / np.abs(F0).max(), color=AQUA)
a2.axhline(0, color=INK2, lw=0.8)
stag = a2.axvline(r0 / c.AU, color=INK2, ls="--", lw=1)
a2.set_ylim(-1.2, 1.2); a2.set_xlabel("r [AU]")
a2.set_ylabel(r"$\dot M(r)\,/\,\max|\dot M|$" "\n" r"+ inward, $-$ outward")
fig.tight_layout()

def draw(k):
    S = frames[k]
    lM.set_ydata(dMdlnr(S)); lJ.set_ydata(dJdlnr(S))
    lA.set_ydata(dMdlnr(v.ring_solution(g.r, taus[k] * t_nu, m_ring, r0, nu0)))
    F = mdot_faces(S, g, nu_ring)
    lF.set_ydata(F / np.abs(F).max())
    # TODO: the stagnation radius r_st in AU, where F changes from inward (> 0) to outward (<= 0).
    #       If F crosses more than once, take the crossing with the largest jump in F.
    r_st = np.nan
    stag.set_xdata([r_st, r_st])
    txt.set_text(fr"$\tau$ = {taus[k]:.3f}   (t = {taus[k] * t_nu / c.YR:,.0f} yr)")
    fr.set_text(f"accreted: {1 - (S * g.area).sum() / M0:5.1%} of mass, "
                f"{1 - (S * g.area * j).sum() / J0:5.1%} of J;  stagnation radius {r_st:.1f} AU")
    return lM, lJ, lA, lF, stag, txt, fr

anim = animation.FuncAnimation(fig, draw, frames=len(taus), interval=80)
plt.close(fig)
show(anim, "spreading_ring", fps=12)

**Question A.** By $\tau = 10$, what fractions of the ring's mass and of its angular momentum have reached the star? Explain in one or two sentences why the two fractions differ.

*Your answer:*

## 3. The long-time limit

The analytic solution can be followed much longer than the grid. Track three quantities to $\tau = 1000$, each as a fraction of the ring's initial mass or angular momentum: the mass that has reached the star, the angular momentum outside $2r_0$, and the mass outside $2r_0$.

**TODO 6.**

In [ ]:
rr = np.geomspace(1e-6, 1e5, 60000) * r0
jj = np.sqrt(c.G * c.M_SUN * rr)
ann = np.pi * np.diff(np.concatenate(([0], np.sqrt(rr[1:] * rr[:-1]), [rr[-1] * 1.0002])) ** 2)
J_ring = m_ring * np.sqrt(c.G * c.M_SUN * r0)
tau_long = np.geomspace(1e-2, 1e3, 120)
f_acc, f_j_out, f_m_out = [], [], []
for tk in tau_long:
    S = v.ring_solution(rr, tk * t_nu, m_ring, r0, nu0)
    m_i = S * ann                     # mass in each annulus
    J_i = m_i * jj                    # angular momentum in each annulus
    # TODO: append, as fractions of the ring's INITIAL mass m_ring and angular momentum J_ring,
    #       the mass that has reached the star (whatever is no longer on the grid), the angular
    #       momentum outside 2 r0, and the mass outside 2 r0.
    raise NotImplementedError("TODO: long-time fractions")

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.semilogx(tau_long, f_acc, color=BLUE, label="mass accreted by the star")
ax.semilogx(tau_long, f_j_out, color=ORANGE, label=r"angular momentum outside $2r_0$")
ax.semilogx(tau_long, f_m_out, color=ORANGE, ls="--", lw=1.4, label=r"mass outside $2r_0$")
ax.set_xlabel(r"$\tau = 12\nu t / r_0^2$"); ax.set_ylabel("fraction of total")
ax.set_ylabel("fraction of initial total")
ax.set_ylim(0, 1.02); ax.legend(fontsize=9, loc="upper left"); fig.tight_layout()

**Question B.** At $\tau = 1000$, what fraction of the angular momentum is outside $2r_0$, and what fraction of the mass? How do the two curves behave as $\tau$ grows, and what does that say about where the mass and the angular momentum end up as $\tau \to \infty$?

*Your answer:*

## 4. An $\alpha$ disk over 10 Myr

This is the run of Problem 4(c): $\alpha = 10^{-3}$, $T = 280\,(r/{\rm AU})^{-1/2}$ K, $r_1 = 30$ AU and $M_0 = 0.05\,M_\odot$. The animation shows $\Sigma(r)$ against the analytic solution, the mass flow normalized to the accretion rate onto the star, and $M(t)$ and $\dot M_\star(t)$ against the expressions in the handout. The dashed line marks the analytic radius where the flow reverses, $r_1\tilde t/2$ with $\tilde t = 1 + t/t_s$. Use this run and its figure for your answer to 4(c).

**TODO 7.**

In [ ]:
alpha, r1, M0d = 1e-3, 30 * c.AU, 0.05 * c.M_SUN
# TODO: the run of Problem 4(c). Build the grid (0.1 to 1e4 AU, 20 cells per decade), the alpha
#       viscosity nud on it, nu1 = nu(r1) and t_s; start from similarity_solution at t = 0 and call
#       v.evolve to 10 Myr with n_out = 100 snapshots. Keep the names used below.
gd = nud = nu1 = t_s = None
t_d, sig_d, mass_d = None, None, None
raise NotImplementedError("TODO: alpha disk")
mdot_star = np.array([mdot_faces(S_, gd, nud)[0] for S_ in sig_d])      # through the inner edge
print(f"t_s = {t_s / c.MYR:.2f} Myr")

In [ ]:
SIG_UNIT = 10.0                      # kg/m^2 -> g/cm^2
yr_msun = c.YR / c.M_SUN
xd, tm, Tt = gd.r / c.AU, t_d / c.MYR, 1 + t_d / t_s
fig = plt.figure(figsize=(10, 6.2))
gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], height_ratios=[1.4, 1])
aS = fig.add_subplot(gs[0, 0]); aF = fig.add_subplot(gs[1, 0], sharex=aS)
aM = fig.add_subplot(gs[0, 1]); aR = fig.add_subplot(gs[1, 1], sharex=aM)

aS.loglog(xd, sig_d[0] / SIG_UNIT, color=INK2, lw=1, alpha=0.5, label="t = 0")
lS, = aS.loglog(xd, sig_d[0] / SIG_UNIT, color=BLUE, label="step()")
lSa, = aS.loglog(xd, sig_d[0] / SIG_UNIT, ":", color=INK, lw=1.2, label="analytic")
aS.set_xlim(0.1, 3000); aS.set_ylim(1e-3, 3e3)
aS.set_ylabel(r"$\Sigma$ [g cm$^{-2}$]"); aS.legend(fontsize=9, loc="lower left")
tS = aS.text(0.97, 0.95, "", transform=aS.transAxes, ha="right", va="top", color=INK)
lF, = aF.semilogx(gd.r_face / c.AU, mdot_faces(sig_d[0], gd, nud) / mdot_star[1], color=AQUA)
aF.axhline(0, color=INK2, lw=0.8)
rrev = aF.axvline(r1 / 2 / c.AU, color=INK2, ls="--", lw=1)
aF.set_ylim(-0.6, 1.3); aF.set_xlabel("r [AU]")
aF.set_ylabel(r"$\dot M(r)\,/\,\dot M_\star(t)$" "\n" r"+ inward")
aM.plot(tm, M0d * Tt ** -0.5 / c.M_SUN, ":", color=INK, lw=1.2)
aM.plot(tm, mass_d / c.M_SUN, color=BLUE, lw=1, alpha=0.35)
lMt, = aM.plot(tm[:1], mass_d[:1] / c.M_SUN, color=BLUE)
aM.set_ylabel(r"disk mass [$M_\odot$]"); aM.set_ylim(0, 0.055)
aR.semilogy(tm, 3 * nu1 * M0d / (2 * r1 ** 2) * Tt ** -1.5 * yr_msun, ":", color=INK, lw=1.2, label="analytic")
aR.semilogy(tm[1:], mdot_star[1:] * yr_msun, color=AQUA, lw=1, alpha=0.35)
lRt, = aR.semilogy(tm[1:2], mdot_star[1:2] * yr_msun, color=AQUA, label="onto the star")
aR.set_xlabel("t [Myr]"); aR.set_ylabel(r"$\dot M_\star$ [$M_\odot$ yr$^{-1}$]")
aR.set_ylim(3e-10, 5e-8); aR.legend(fontsize=9)
fig.tight_layout()

def draw_d(k):
    S = sig_d[k]; kk = max(k, 1)
    lS.set_ydata(S / SIG_UNIT)
    lSa.set_ydata(v.similarity_solution(gd.r, t_d[k], M0d, r1, nu1) / SIG_UNIT)
    lF.set_ydata(mdot_faces(S, gd, nud) / mdot_star[kk])
    r_rev = r1 * Tt[k] / 2 / c.AU
    rrev.set_xdata([r_rev, r_rev])
    lMt.set_data(tm[:k + 1], mass_d[:k + 1] / c.M_SUN)
    lRt.set_data(tm[1:kk + 1], mdot_star[1:kk + 1] * yr_msun)
    tS.set_text(f"t = {tm[k]:.1f} Myr\nM = {mass_d[k] / c.M_SUN:.3f} $M_\\odot$")
    return lS, lSa, lF, rrev, lMt, lRt, tS

anim_d = animation.FuncAnimation(fig, draw_d, frames=len(t_d), interval=90)
plt.close(fig)
show(anim_d, "alpha_disk", fps=10)

## 5. Viscous timescales

The time to rearrange the disk over a radius $r$ is $t_\nu = r^2/\nu$. Compare it with the 1–3 Myr lifetimes of observed disks.

**TODO 8.**

In [ ]:
r_ax = np.geomspace(0.1, 300, 200) * c.AU
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.axhspan(1, 3, color=GRID, alpha=0.8, lw=0)
ax.text(0.13, 1.6, "observed disk lifetimes, 1–3 Myr", color=INK2, fontsize=9)
for a_, col in [(1e-4, AQUA), (1e-3, BLUE), (1e-2, ORANGE)]:
    tnu = ...      # TODO: viscous time r^2 / nu in Myr at each radius in r_ax
    raise NotImplementedError("TODO: viscous time")
    ax.loglog(r_ax / c.AU, tnu, color=col, label=fr"$\alpha = 10^{{{int(np.log10(a_))}}}$")
ax.set_xlabel("r [AU]"); ax.set_ylabel(r"$t_\nu = r^2/\nu$ [Myr]")
ax.legend(fontsize=9, loc="upper left"); fig.tight_layout()

**Question C.** Which of the three values of $\alpha$, applied to the whole disk, is consistent with disks surviving 1–3 Myr out to tens of AU? One sentence.

*Your answer:*